Scenario: The Field Marketing Lead has asked you to help prepare their team for conference season. You create a Conference Prep Agent with IBM BeeAI that routes each query across web search, Wikipedia lookup, and RAG over a local vector store of internal notes, returning a synthesized answer with sources.

## Install  BeeAI Framework and set up our environment.



In [ ]:
import sys
if sys.version_info < (3, 11):
    raise RuntimeError(
        f"Python {sys.version.split()[0]} is too old for BeeAI. "
        "Select the project kernel .venv/bin/python (Python 3.13), then run this notebook again."
    )

%pip install -q uv
!uv pip install -q --python "{sys.executable}" arize-phoenix arize-phoenix-otel s3fs fsspec==2025.3.0 \
 gcsfs nest_asyncio openinference-instrumentation-beeai jedi \
 beeai-framework beeai-framework[duckduckgo] beeai-framework[llama_index] beeai-framework[wikipedia] \
 langchain langchain_community wikipedia unstructured markdown requests==2.32.4 "graphql-core==3.2.6"

![ -d beeai_framework_synthetic_data ] || git clone https://github.com/sandijean90/beeai_framework_synthetic_data.git

from IPython.display import HTML, display
def set_css(*args, **kwargs):
    display(HTML("\n<style>\n pre{\n white-space: pre-wrap;\n}\n</style>\n"))
get_ipython().events.register("pre_run_cell", set_css)

Import modules:


In [ ]:
import os
from datetime import date

import phoenix as px
from phoenix.otel import register
from dotenv import load_dotenv

from beeai_framework.agents.experimental import RequirementAgent
from beeai_framework.agents.experimental.requirements.conditional import ConditionalRequirement
from beeai_framework.backend import ChatModel, ChatModelParameters
from beeai_framework.backend.document_loader import DocumentLoader
from beeai_framework.backend.embedding import EmbeddingModel
from beeai_framework.backend.text_splitter import TextSplitter
from beeai_framework.backend.vector_store import VectorStore
from beeai_framework.memory import SummarizeMemory
from beeai_framework.middleware.trajectory import GlobalTrajectoryMiddleware
from beeai_framework.tools import Tool, tool
from beeai_framework.tools.search.duckduckgo import DuckDuckGoSearchTool
from beeai_framework.tools.search.retrieval import VectorStoreSearchTool
from beeai_framework.tools.search.wikipedia import WikipediaTool, WikipediaToolInput
from beeai_framework.tools.think import ThinkTool
from openinference.instrumentation.beeai import BeeAIInstrumentor

### Choose LLM (Azure OpenAI)

In [ ]:
load_dotenv()

provider = "azure_openai"
model = os.environ.get("AZURE_OPENAI_CHAT_MODEL", "gpt-5-mini")

llm = ChatModel.from_name(f"{provider}:{model}", ChatModelParameters(temperature=1))

### System Prompt

In [ ]:
instruct_prompt = """You help field marketing teams prep for conferences by answering questions on companies that they need to prepare to talk to. You produce quick and actionable briefs, doing your best to answer the user's question.

Today's date is {todays_date}.

Tools:
- ThinkTool: Helps you plan and reason before you act. Use this tool when you need to think.
- DuckDuckGoSearchTool: Use this tool to collect current information on agendas, speakers, news, competitor moves. Include title + date + link to the resources you find in your answer. Do not use this tool for internal notes or artifacts.
- wikipedia_tool: Use this tool to get company/org history (not for breaking news). Only look up company names as the input.
- internal_document_search: past meetings, playbooks, artifacts. If you use information from this in your response, label it as [Internal]. Always use this tool when internal notes or content are referenced.

Basic Rules:
- Be concise and practical.
- Return a synthesized answer with sources: title, date, and link for web results, and [Internal] plus the note title for internal notes.
- Favor recent info (agenda/news ≤30 days; exec changes/funding ≤180 days); flag older items.
- If you don't know, say so. Don't make things up.
"""

In [ ]:
todays_date = date.today().strftime("%B %d, %Y")
formatted_instruct_prompt = instruct_prompt.format(
        todays_date=todays_date)

Memory:

In [ ]:
memory = SummarizeMemory(llm)

### Tools

In [ ]:
think_tool = ThinkTool()

The **DuckDuckGoSearchTool** is a Web Search tool that provides relevant data from the internet to the LLM


In [ ]:
search_tool = DuckDuckGoSearchTool()

In [ ]:
@tool
async def wikipedia_tool(query: str) -> str:
    """Search Wikipedia for factual and historical information.

    Args:
        query: The topic or company name to look up.

    Returns:
        Text content from the matching Wikipedia article.
    """
    wiki = WikipediaTool(language="en")
    response = await wiki.run(WikipediaToolInput(query=query, full_text=False))
    return response.get_text_content()

### RAG

In [ ]:
import shutil, subprocess, time, socket

def _ollama_up(timeout=1):
    try:
        socket.create_connection(("127.0.0.1", 11434), timeout=timeout).close()
        return True
    except OSError:
        return False

def ensure_ollama():
    if _ollama_up():
        return
    if not shutil.which("ollama"):
        raise RuntimeError("ollama is not installed")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    for _ in range(60):
        if _ollama_up():
            return
        time.sleep(1)
    raise RuntimeError("ollama did not become reachable on :11434 within 60s")

ensure_ollama()
!ollama pull nomic-embed-text:latest
embedding_model = EmbeddingModel.from_name("ollama:nomic-embed-text")

In [ ]:
# Internal notes: local copy, or the Colab clone of the synthetic-data repo.
from pathlib import Path

def find_notes():
    candidates = [
        Path("data/rag_conference_prep_agent.txt"),
        Path("/content/beeai_framework_synthetic_data/rag_conference_prep_agent.txt"),
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("rag_conference_prep_agent.txt")

notes_path = find_notes()
with open(notes_path, "r") as file:
    content = file.read()
print(content)

In [ ]:
loader = DocumentLoader.from_name(
    name="langchain:UnstructuredMarkdownLoader", file_path=str(notes_path)
)
try:
    documents = await loader.load()
except Exception as e:
    print(f"Failed to load documents: {e}")
    raise

# Split documents into chunks
text_splitter = TextSplitter.from_name(
    name="langchain:RecursiveCharacterTextSplitter", chunk_size=1000, chunk_overlap=200
)
documents = await text_splitter.split_documents(documents)
print(f"Loaded {len(documents)} document chunks")

In [ ]:
# Create vector store and add documents
ensure_ollama()
vector_store = VectorStore.from_name(name="beeai:TemporalVectorStore", embedding_model=embedding_model)
await vector_store.add_documents(documents=documents)
print("Vector store populated with documents")


In [ ]:
# Create the vector store search tool
internal_document_search = VectorStoreSearchTool(vector_store=vector_store)

### Control Agent Behavior

In [ ]:
requirement_1 = ConditionalRequirement(ThinkTool, consecutive_allowed=False, force_at_step=1 )

requirement_2 = ConditionalRequirement(wikipedia_tool, only_after=ThinkTool, consecutive_allowed=True, priority=10,)

requirement_3 = ConditionalRequirement(DuckDuckGoSearchTool, only_after=ThinkTool, consecutive_allowed=True, min_invocations=1, max_invocations=3 ,priority=15,)

requirement_4 = ConditionalRequirement(internal_document_search, only_after=ThinkTool, consecutive_allowed=True, min_invocations=1, priority=20,)


In [ ]:
load_dotenv()

px_session = px.launch_app()

tracer_provider = register(
    project_name="conference-prep-agent",
    endpoint="http://localhost:6006/v1/traces",
    batch=True,
)
BeeAIInstrumentor().instrument(tracer_provider=tracer_provider)

In [ ]:
print(f"Phoenix UI: {px_session.url}")

##  Assemble Agent

In [ ]:
agent = RequirementAgent(
    llm=llm,
    instructions=formatted_instruct_prompt,
    memory=memory,
    tools=[ThinkTool(), DuckDuckGoSearchTool(), wikipedia_tool, internal_document_search],
    requirements=[requirement_1, requirement_2, requirement_3, requirement_4],
    middlewares=[GlobalTrajectoryMiddleware(included=[Tool])],
)

### Run Agent

In [ ]:
question = "I'm planning on meeting the Moderna rep at the next conference. Give me a one pager and remind me where we left off on previous internal discussions."

response = await agent.run(question, max_retries_per_step=3, total_max_retries=25, max_iterations=30)
print(response.last_message.text)

In [ ]:
px_session.view()